In [ ]:
import pandas as pd
quali_df =  pd.read_csv('data/2025_qualifying_results.csv')
races_df = pd.read_csv('data/2025_race_results.csv')
sprint_quali_df = pd.read_csv('data/2025_sprint_qualifying_results.csv')
sprint_df = pd.read_csv('data/2025_sprint_race_results.csv')

In [ ]:
races_df

,driver_number,abbreviation,full_name,team_name,grid_position,race_position,points,status,race_number,race_name,race_date
0,4,NOR,Lando Norris,McLaren,1.0,1,25.0,Finished,1,Australian Grand Prix,2025-03-16
1,1,VER,Max Verstappen,Red Bull Racing,3.0,2,18.0,Finished,1,Australian Grand Prix,2025-03-16
2,63,RUS,George Russell,Mercedes,4.0,3,15.0,Finished,1,Australian Grand Prix,2025-03-16
3,12,ANT,Kimi Antonelli,Mercedes,16.0,4,12.0,Finished,1,Australian Grand Prix,2025-03-16
4,23,ALB,Alexander Albon,Williams,6.0,5,10.0,Finished,1,Australian Grand Prix,2025-03-16
...,...,...,...,...,...,...,...,...,...,...,...
474,23,ALB,Alexander Albon,Williams,17.0,16,0.0,Finished,24,Abu Dhabi Grand Prix,2025-12-07
475,6,HAD,Isack Hadjar,Racing Bulls,9.0,17,0.0,Lapped,24,Abu Dhabi Grand Prix,2025-12-07
476,30,LAW,Liam Lawson,Racing Bulls,13.0,18,0.0,Lapped,24,Abu Dhabi Grand Prix,2025-12-07
477,10,GAS,Pierre Gasly,Alpine,19.0,19,0.0,Lapped,24,Abu Dhabi Grand Prix,2025-12-07


In [ ]:
def inverted_score(df_value, max_value):
    return ((1 - df_value / max_value) * 10)

def higher_score(df_value, max_value):
    return ((df_value / max_value) * 10)

def min_max_score(df_value,min_value,max_value):
    return ((df_value-min_value) / (max_value-min_value) * 10)

In [ ]:
#TOTAL points per driver
drivers_performance = pd.concat([races_df[['abbreviation','driver_number','full_name','points']],sprint_df[['abbreviation','driver_number','full_name','points']]],ignore_index = True)

drivers_performance = (
    drivers_performance
    .groupby(['abbreviation', 'driver_number','full_name'], as_index=False)['points']
    .sum()
    .sort_values(by='points', ascending=False)
)
drivers_performance.rename(columns={'points':'total_points'},inplace=True)
#normalizing 
max_point = drivers_performance['total_points'].max()
drivers_performance['score_points'] = higher_score(drivers_performance['total_points'] , max_point)
drivers_performance['score_points'] = drivers_performance['score_points'].round(2)
drivers_performance

,abbreviation,driver_number,full_name,total_points,score_points
13,NOR,4,Lando Norris,423.0,10.00
20,VER,1,Max Verstappen,421.0,9.95
15,PIA,81,Oscar Piastri,410.0,9.69
16,RUS,63,George Russell,319.0,7.54
12,LEC,16,Charles Leclerc,242.0,5.72
9,HAM,44,Lewis Hamilton,156.0,3.69
2,ANT,12,Kimi Antonelli,150.0,3.55
0,ALB,23,Alexander Albon,73.0,1.73
17,SAI,55,Carlos Sainz,64.0,1.51
1,ALO,14,Fernando Alonso,56.0,1.32


In [ ]:
#finishing position metrics
finish_data = races_df.copy()

finish_data['race_position'] = pd.to_numeric(finish_data['race_position'],errors='coerce')


finished_races = finish_data[~finish_data['status'].isin(['Retired','Did not start', 'Disqualified'])].copy()

races_finish_metrics = (
    finished_races.groupby(['abbreviation', 'driver_number', 'full_name'],as_index=False)  
    .agg(
        races_avg_finish=('race_position', 'mean'),
        races_finish_std=('race_position', 'std'),
        races_best_finish=('race_position', 'min'),
        races_worst_finish=('race_position', 'max'),
        races_finished=('race_position', 'count')
    )
)

races_finish_metrics['races_avg_finish'] = races_finish_metrics['races_avg_finish'].round(2)
races_finish_metrics['races_finish_std'] = races_finish_metrics['races_finish_std'].round(2)

races_finish_metrics = races_finish_metrics.sort_values(
    by='races_avg_finish',
    ascending=True
)
races_finish_metrics

sprint_finish_data = sprint_df.copy()

sprint_finish_data['sprint_position'] = pd.to_numeric(sprint_finish_data['sprint_position'],errors='coerce')

sprint_finished_races = sprint_finish_data[~sprint_finish_data['status'].isin(['Retired','Did not start', 'Disqualified'])].copy()

sprint_finish_metrics = sprint_finished_races.groupby(['abbreviation', 'driver_number', 'full_name'],as_index=False).agg(
    sprint_avg_finish=('sprint_position', 'mean'),
    sprint_finish_std=('sprint_position', 'std'),
    sprint_best_finish=('sprint_position', 'min'),
    sprint_worst_finish=('sprint_position', 'max'),
    sprint_races_finished=('sprint_position', 'count')
)

sprint_finish_metrics['sprint_avg_finish'] = sprint_finish_metrics['sprint_avg_finish'].round(2)
sprint_finish_metrics['sprint_finish_std'] = sprint_finish_metrics['sprint_finish_std'].round(2)

sprint_finish_metrics = sprint_finish_metrics.sort_values(
    by='sprint_avg_finish',
    ascending=True
)

finish_metrics = races_finish_metrics.merge(
    sprint_finish_metrics,
    on= ['abbreviation', 'driver_number', 'full_name'],
    how= 'left'
)
finish_metrics = finish_metrics.sort_values(by='races_avg_finish',ascending=True).reset_index(drop=True)
finish_metrics

#normalizing for races_finish_avg
max_value_avg = finish_metrics['races_avg_finish'].max()

finish_metrics['score_races_avg_finish'] = (1 - finish_metrics['races_avg_finish'] / max_value_avg) * 10
finish_metrics['score_races_avg_finish'] = finish_metrics['score_races_avg_finish'].round(2)

#normalizing for races_finish_std
max_value_std = finish_metrics['races_finish_std'].max()
finish_metrics['score_races_finish_std'] = inverted_score(finish_metrics['races_finish_std'] , max_value_std)
finish_metrics['score_races_finish_std'] = finish_metrics['score_races_finish_std'].round(2)
finish_metrics

,abbreviation,driver_number,full_name,races_avg_finish,races_finish_std,races_best_finish,races_worst_finish,races_finished,sprint_avg_finish,sprint_finish_std,sprint_best_finish,sprint_worst_finish,sprint_races_finished,score_races_avg_finish,score_races_finish_std
0,NOR,4,Lando Norris,2.24,1.45,1.0,7.0,21,3.20,2.86,1.0,8.0,5,8.58,7.09
1,PIA,81,Oscar Piastri,2.77,1.97,1.0,9.0,22,1.75,0.50,1.0,2.0,4,8.25,6.04
2,VER,1,Max Verstappen,3.04,2.51,1.0,10.0,23,5.00,6.03,1.0,17.0,6,8.08,4.96
3,RUS,63,George Russell,4.54,2.50,1.0,11.0,24,4.50,3.78,2.0,12.0,6,7.13,4.98
4,LEC,16,Charles Leclerc,5.05,2.87,2.0,14.0,21,6.40,3.71,4.0,13.0,5,6.81,4.24
5,HAM,44,Lewis Hamilton,7.00,2.37,4.0,12.0,21,7.83,6.65,1.0,17.0,6,5.58,5.24
6,ANT,12,Kimi Antonelli,8.20,4.98,2.0,18.0,20,7.83,4.96,2.0,17.0,6,4.82,0.00
7,ALB,23,Alexander Albon,9.65,3.63,5.0,16.0,20,11.83,4.07,6.0,17.0,6,3.90,2.71
8,ALO,14,Fernando Alonso,10.26,3.54,5.0,17.0,19,9.25,3.59,6.0,14.0,4,3.51,2.89
9,SAI,55,Carlos Sainz,10.42,3.89,3.0,18.0,19,9.60,5.77,3.0,17.0,5,3.41,2.19


In [ ]:
#qualifying metrics
races_quali_data = quali_df.copy()
races_quali_data['quali_position'] = pd.to_numeric(races_quali_data['quali_position'],errors='coerce')

races_quali_metrics = (
    races_quali_data.groupby(['abbreviation', 'driver_number', 'full_name'],as_index=False)
    .agg(
        avg_quali_position = ('quali_position','mean'),
        best_quali_position=('quali_position', 'min'),
        worst_quali_position=('quali_position', 'max'),
        quali_sessions=('quali_position', 'count')
    )
) 

races_quali_metrics['avg_quali_position'] = (races_quali_metrics['avg_quali_position'].round(2))

races_quali_metrics = races_quali_metrics.sort_values(by='avg_quali_position',ascending=True).reset_index(drop=True)

races_quali_metrics

sprint_quali_data = sprint_quali_df.copy()
sprint_quali_data['sprint_quali_position'] = pd.to_numeric(sprint_quali_data['sprint_quali_position'],errors='coerce') 

sprint_quali_metrics = (
    sprint_quali_data.groupby(['abbreviation', 'driver_number', 'full_name'],as_index=False)
    .agg(
        avg_sprint_quali=('sprint_quali_position', 'mean'),
        best_sprint_quali=('sprint_quali_position', 'min'),
        worst_sprint_quali=('sprint_quali_position', 'max'),
        sprint_quali_sessions=('sprint_quali_position', 'count')
    )
)
sprint_quali_metrics= sprint_quali_metrics.sort_values(by='avg_sprint_quali',ascending=True).reset_index(drop=True)
sprint_quali_metrics

quali_metrics = races_quali_metrics.merge(
    sprint_quali_metrics,
    on=['abbreviation', 'driver_number', 'full_name'],
    how='left'
)
quali_metrics = quali_metrics.sort_values(by='avg_quali_position',ascending=True).reset_index(drop=True)
max_value_avg_quali = quali_metrics['avg_quali_position'].max()
quali_metrics['score_avg_quali_position'] = inverted_score(quali_metrics['avg_quali_position'],max_value_avg_quali)
quali_metrics['score_avg_quali_position'] = quali_metrics['score_avg_quali_position'].round(2)
quali_metrics

,abbreviation,driver_number,full_name,avg_quali_position,best_quali_position,worst_quali_position,quali_sessions,avg_sprint_quali,best_sprint_quali,worst_sprint_quali,sprint_quali_sessions,score_avg_quali_position
0,NOR,4,Lando Norris,2.96,1.0,10.0,24,3.000000,1.0,6.0,6,8.26
1,PIA,81,Oscar Piastri,3.04,1.0,9.0,24,2.166667,1.0,3.0,6,8.21
2,VER,1,Max Verstappen,3.50,1.0,16.0,24,3.500000,1.0,6.0,6,7.94
3,RUS,63,George Russell,4.38,1.0,14.0,24,5.666667,2.0,13.0,6,7.42
4,LEC,16,Charles Leclerc,5.46,1.0,11.0,24,6.833333,4.0,10.0,6,6.79
5,ANT,12,Kimi Antonelli,8.58,2.0,18.0,24,8.000000,1.0,20.0,6,4.95
6,HAM,44,Lewis Hamilton,9.04,3.0,20.0,24,10.500000,1.0,18.0,6,4.68
7,HAD,6,Isack Hadjar,9.67,4.0,20.0,24,10.833333,9.0,15.0,6,4.31
8,ALO,14,Fernando Alonso,10.38,5.0,19.0,24,8.333333,4.0,14.0,6,3.89
9,SAI,55,Carlos Sainz,11.00,2.0,20.0,24,11.500000,6.0,20.0,6,3.53


In [ ]:
#qualifying vs race performance GAP
race_position_data = races_df.copy()

# Convert positions to numeric
race_position_data[['grid_position','race_position']] = race_position_data[['grid_position','race_position']].apply(pd.to_numeric,errors='coerce')

# Use only drivers who officially finished 
race_position_data = race_position_data[race_position_data['status'] == 'Finished'].copy()

# Use only drivers who officially finished
race_position_data['quali_race_gap'] = race_position_data['grid_position'] - race_position_data['race_position']

# Calculate metrics per driver
races_finish_gap_metrics = race_position_data.groupby(
    ['abbreviation', 'driver_number', 'full_name'],as_index=False
    ).agg(
        avg_quali_race_gap = ('quali_race_gap','mean'),
        races_used = ('quali_race_gap','count')
        )
races_finish_gap_metrics['avg_quali_race_gap'] = races_finish_gap_metrics['avg_quali_race_gap'].round(2)
races_finish_gap_metrics = races_finish_gap_metrics.sort_values(
        by='avg_quali_race_gap',
        ascending=False
    ).reset_index(drop=True)


races_finish_gap_metrics

sprint_position_data = sprint_df.copy()

# Convert positions to numeric
sprint_position_data[['grid_position', 'sprint_position']] = (sprint_position_data[['grid_position', 'sprint_position']].apply(pd.to_numeric, errors='coerce'))

# Use only drivers who officially finished
sprint_position_data = sprint_position_data[sprint_position_data['status'] == 'Finished'].copy()

# Calculate positions gained/lost
sprint_position_data['sprint_quali_race_gap'] = (sprint_position_data['grid_position'] - sprint_position_data['sprint_position'])

# Calculate metrics per driver
sprint_finish_gap_metrics = (
    sprint_position_data.groupby(['abbreviation', 'driver_number', 'full_name'],as_index=False
    ).agg(
        avg_sprint_quali_race_gap=('sprint_quali_race_gap', 'mean'),
        sprints_used=('sprint_quali_race_gap', 'count')
    )
)
sprint_finish_gap_metrics['avg_sprint_quali_race_gap'] = sprint_finish_gap_metrics['avg_sprint_quali_race_gap'].round(2)
sprint_finish_gap_metrics = sprint_finish_gap_metrics.sort_values(by='avg_sprint_quali_race_gap',ascending=False).reset_index(drop=True)
# Sort by biggest average positions gained
sprint_finish_gap_metrics = (sprint_finish_gap_metrics.sort_values(
        by='avg_sprint_quali_race_gap',
        ascending=False
    )
    .reset_index(drop=True)
)

sprint_finish_gap_metrics

quali_finish_gap_metrics = races_finish_gap_metrics.merge(
    sprint_finish_gap_metrics,
    on=['abbreviation', 'driver_number', 'full_name'],
    how='left'
)
quali_finish_gap_metrics = quali_finish_gap_metrics.sort_values(by = 'avg_quali_race_gap',ascending=False).reset_index(drop=True)
max_value_quali_race = quali_finish_gap_metrics['avg_quali_race_gap'].max()
min_value_quali_race = quali_finish_gap_metrics['avg_quali_race_gap'].min()
quali_finish_gap_metrics['score_avg_quali_race_gap'] = min_max_score(quali_finish_gap_metrics['avg_quali_race_gap'],min_value_quali_race,max_value_quali_race)
quali_finish_gap_metrics['score_avg_quali_race_gap'] = quali_finish_gap_metrics['score_avg_quali_race_gap'].round(2)
quali_finish_gap_metrics

,abbreviation,driver_number,full_name,avg_quali_race_gap,races_used,avg_sprint_quali_race_gap,sprints_used,score_avg_quali_race_gap
0,HUL,27,Nico Hulkenberg,5.15,13,-2.33,6,10.00
1,BEA,87,Oliver Bearman,3.28,18,1.00,6,6.71
2,STR,18,Lance Stroll,3.08,12,2.00,5,6.36
3,ALB,23,Alexander Albon,3.06,16,-1.17,6,6.32
4,OCO,31,Esteban Ocon,3.00,16,1.80,5,6.21
5,HAM,44,Lewis Hamilton,2.65,20,2.67,6,5.60
6,DOO,7,Jack Doohan,2.00,3,-1.50,2,4.45
7,TSU,22,Yuki Tsunoda,1.19,16,5.67,6,3.03
8,COL,43,Franco Colapinto,1.00,5,1.33,3,2.69
9,HAD,6,Isack Hadjar,0.85,13,0.50,6,2.43


In [ ]:
#DNF (Did Not Finish) metrics
races_dnf_data = races_df.copy()

#mark DNFs
races_dnf_data['dnf'] = (races_dnf_data['status'] == 'Retired').astype(int)
#count dns and dsq
races_dnf_metrics = races_dnf_data.groupby(
    ['abbreviation', 'driver_number', 'full_name'],as_index=False
    ).agg(
        race_entries = ('status','size'),
        dns_count = ('status', lambda x: (x == 'Did not start').sum()),
        dsq_count = ('status', lambda x: (x == 'Disqualified').sum()),
        dnf_count = ('dnf','sum')

    )

races_dnf_metrics['races_started'] = (
    races_dnf_metrics['race_entries'] - races_dnf_metrics['dns_count']
)

races_dnf_metrics['dnf_rate'] = (
    races_dnf_metrics['dnf_count'] / races_dnf_metrics['races_started']
) * 100
races_dnf_metrics['dnf_rate'] = races_dnf_metrics['dnf_rate'].round(2)
races_dnf_metrics = races_dnf_metrics.sort_values(by='dnf_rate',ascending=True).reset_index(drop=True)
races_dnf_metrics

# DNF (Did Not Finish) metrics - SPRINT RACES

sprint_dnf_data = sprint_df.copy()

# Mark DNFs
sprint_dnf_data['dnf'] = (sprint_dnf_data['status'] == 'Retired').astype(int)


# Calculate Sprint DNF metrics
sprint_dnf_metrics = (
    sprint_dnf_data.groupby(['abbreviation', 'driver_number', 'full_name'],as_index=False
    )
    .agg(
        sprint_entries=('status', 'size'),
        sprint_dns_count=('status',lambda x: (x == 'Did not start').sum()),
        sprint_dsq_count=('status',lambda x: (x == 'Disqualified').sum()),
        sprint_dnf_count=('dnf', 'sum')
    )
)

# Calculate Sprint races started
sprint_dnf_metrics['sprints_started'] = (
    sprint_dnf_metrics['sprint_entries'] - sprint_dnf_metrics['sprint_dns_count']
)


# Calculate DNF rate
sprint_dnf_metrics['sprint_dnf_rate'] = (
    sprint_dnf_metrics['sprint_dnf_count'] / sprint_dnf_metrics['sprints_started']
) * 100


# Round DNF rate
sprint_dnf_metrics['sprint_dnf_rate'] = (sprint_dnf_metrics['sprint_dnf_rate'].round(2))


# Sort by lowest DNF rate
sprint_dnf_metrics = (sprint_dnf_metrics.sort_values(by='sprint_dnf_rate', ascending=True)
    .reset_index(drop=True)
)

sprint_dnf_metrics

dnf_metrics = races_dnf_metrics.merge(
    sprint_dnf_metrics,
    on=['abbreviation', 'driver_number', 'full_name'],
    how='left'
)
dnf_metrics = dnf_metrics.sort_values(by='dnf_rate',ascending=True).reset_index(drop=True)
max_value_dnf_rate = dnf_metrics['dnf_rate'].max()
dnf_metrics['score_dnf_rate'] = inverted_score(dnf_metrics['dnf_rate'],max_value_dnf_rate)
dnf_metrics['score_dnf_rate'] = dnf_metrics['score_dnf_rate'].round(2)
dnf_metrics

,abbreviation,driver_number,full_name,race_entries,dns_count,dsq_count,dnf_count,races_started,dnf_rate,sprint_entries,sprint_dns_count,sprint_dsq_count,sprint_dnf_count,sprints_started,sprint_dnf_rate,score_dnf_rate
0,COL,43,Franco Colapinto,18,1,0,0,17,0.00,4,0,0,1,4,25.00,10.00
1,RUS,63,George Russell,24,0,0,0,24,0.00,6,0,0,0,6,0.00,10.00
2,PIA,81,Oscar Piastri,24,0,1,1,24,4.17,6,0,0,2,6,33.33,8.75
3,OCO,31,Esteban Ocon,24,0,0,1,24,4.17,6,0,0,1,6,16.67,8.75
4,TSU,22,Yuki Tsunoda,24,0,0,1,24,4.17,6,0,0,0,6,0.00,8.75
5,VER,1,Max Verstappen,24,0,0,1,24,4.17,6,0,0,0,6,0.00,8.75
6,NOR,4,Lando Norris,24,0,1,2,24,8.33,6,0,0,1,6,16.67,7.50
7,GAS,10,Pierre Gasly,24,0,1,2,24,8.33,6,0,0,1,6,16.67,7.50
8,LEC,16,Charles Leclerc,24,0,1,2,24,8.33,6,0,0,1,6,16.67,7.50
9,HAM,44,Lewis Hamilton,24,0,1,2,24,8.33,6,0,0,0,6,0.00,7.50


In [ ]:
#podium metrics
race_podium_data = races_df.copy()

race_podium_data['race_position'] = pd.to_numeric(race_podium_data['race_position'],errors='coerce')
race_podium_data['podium_count'] = (race_podium_data['race_position']<=3).astype(int)


race_podium_metrics = race_podium_data.groupby(
    ['abbreviation', 'driver_number', 'full_name'],as_index=False
    ).agg(
        podium_count = ('podium_count','sum'),
        race_entries = ('status','size'),
        dns_count = ('status', lambda x: (x == 'Did not start').sum()),
    )
#removes DNS from the races
race_podium_metrics['races_participated'] = race_podium_metrics['race_entries'] - race_podium_metrics['dns_count']

#podium rate
race_podium_metrics['podium_rate'] = (race_podium_metrics['podium_count']/race_podium_metrics['races_participated'])*100

race_podium_metrics.drop(
    columns=['race_entries', 'dns_count'],
    inplace=True
)

race_podium_metrics['podium_rate'] = race_podium_metrics['podium_rate'].round(2)
race_podium_metrics = race_podium_metrics.sort_values(by='podium_rate',ascending=False).reset_index(drop=True)
race_podium_metrics

# SPRINT PODIUM METRICS

sprint_podium_data = sprint_df.copy()

sprint_podium_data['sprint_position'] = pd.to_numeric(sprint_podium_data['sprint_position'],errors='coerce')

# Mark podiums
sprint_podium_data['sprint_podium_count'] = (sprint_podium_data['sprint_position'] <= 3).astype(int)

# Calculate metrics
sprint_podium_metrics = sprint_podium_data.groupby(
    ['abbreviation', 'driver_number', 'full_name'],as_index=False
).agg(
    sprint_podium_count=('sprint_podium_count', 'sum'),
    sprint_entries=('status', 'size'),
    sprint_dns_count=('status',lambda x: (x == 'Did not start').sum())
)

# Remove DNS
sprint_podium_metrics['sprints_participated'] = (sprint_podium_metrics['sprint_entries'] - sprint_podium_metrics['sprint_dns_count'])

# Podium rate
sprint_podium_metrics['sprint_podium_rate'] = (sprint_podium_metrics['sprint_podium_count'] / sprint_podium_metrics['sprints_participated']) * 100

# Round
sprint_podium_metrics['sprint_podium_rate'] = (sprint_podium_metrics['sprint_podium_rate'].round(2))

# Remove unnecessary columns
sprint_podium_metrics.drop(
    columns=['sprint_entries', 'sprint_dns_count'],
    inplace=True
)

# Sort
sprint_podium_metrics = (
    sprint_podium_metrics.sort_values(by='sprint_podium_rate', ascending=False).reset_index(drop=True)
)

sprint_podium_metrics

podium_metrics = race_podium_metrics.merge(
    sprint_podium_metrics,
    on=['abbreviation', 'driver_number', 'full_name'],
    how='left'
)
podium_metrics = podium_metrics.sort_values(by='podium_rate',ascending=False).reset_index(drop=True)
max_value_podium_rate = podium_metrics['podium_rate'].max()
podium_metrics['score_podium_rate'] = higher_score(podium_metrics['podium_rate'],max_value_podium_rate)
podium_metrics['score_podium_rate'] = podium_metrics['score_podium_rate'].round(2)
podium_metrics

,abbreviation,driver_number,full_name,podium_count,races_participated,podium_rate,sprint_podium_count,sprints_participated,sprint_podium_rate,score_podium_rate
0,NOR,4,Lando Norris,18,24,75.00,4,6,66.67,10.00
1,PIA,81,Oscar Piastri,16,24,66.67,4,6,66.67,8.89
2,VER,1,Max Verstappen,15,24,62.50,3,6,50.00,8.33
3,RUS,63,George Russell,9,24,37.50,3,6,50.00,5.00
4,LEC,16,Charles Leclerc,7,24,29.17,0,6,0.00,3.89
5,ANT,12,Kimi Antonelli,3,24,12.50,1,6,16.67,1.67
6,SAI,55,Carlos Sainz,2,23,8.70,1,6,16.67,1.16
7,HUL,27,Nico Hulkenberg,1,23,4.35,0,6,0.00,0.58
8,HAD,6,Isack Hadjar,1,24,4.17,0,6,0.00,0.56
9,BOR,5,Gabriel Bortoleto,0,24,0.00,0,6,0.00,0.00


In [ ]:
#win metrics
race_wins_data = races_df.copy()

race_wins_data['race_position'] = pd.to_numeric(race_wins_data['race_position'],errors='coerce')

race_wins_data['win_count'] = (race_wins_data['race_position'] == 1).astype(int)

race_wins_metrics = race_wins_data.groupby(
    ['abbreviation', 'driver_number', 'full_name'],as_index=False
    ).agg(
        wins_count = ('win_count','sum'),
        race_entries = ('status','size'),
        dns_count = ('status', lambda x: (x == 'Did not start').sum())
    )
race_wins_metrics['races_participated'] = race_wins_metrics['race_entries'] - race_wins_metrics['dns_count']

race_wins_metrics['win_rate'] = (race_wins_metrics['wins_count'] / race_wins_metrics['races_participated']) * 100

race_wins_metrics.drop(columns=['race_entries', 'dns_count'],inplace=True)

race_wins_metrics['win_rate'] = race_wins_metrics['win_rate'].round(2)

race_wins_metrics = race_wins_metrics.sort_values(by='win_rate',ascending=False).reset_index(drop=True)

race_wins_metrics

# SPRINT WIN METRICS

sprint_wins_data = sprint_df.copy()

# Convert position to numeric
sprint_wins_data['sprint_position'] = pd.to_numeric(sprint_wins_data['sprint_position'],errors='coerce')

# Mark wins
sprint_wins_data['sprint_win_count'] = (sprint_wins_data['sprint_position'] == 1).astype(int)


# Calculate metrics
sprint_wins_metrics = sprint_wins_data.groupby(
    ['abbreviation', 'driver_number', 'full_name'],as_index=False
).agg(
    sprint_wins_count=('sprint_win_count', 'sum'),
    sprint_entries=('status', 'size'),
    sprint_dns_count=('status',lambda x: (x == 'Did not start').sum())
)


# Calculate sprints participated
sprint_wins_metrics['sprints_participated'] = (sprint_wins_metrics['sprint_entries'] - sprint_wins_metrics['sprint_dns_count'])


# Calculate win rate
sprint_wins_metrics['sprint_win_rate'] = (sprint_wins_metrics['sprint_wins_count'] / sprint_wins_metrics['sprints_participated']) * 100


# Remove unnecessary columns
sprint_wins_metrics.drop(
    columns=['sprint_entries', 'sprint_dns_count'],inplace=True
)


# Round win rate
sprint_wins_metrics['sprint_win_rate'] = (
    sprint_wins_metrics['sprint_win_rate'].round(2)
)


# Sort by highest win rate
sprint_wins_metrics = (
    sprint_wins_metrics.sort_values(by='sprint_win_rate', ascending=False).reset_index(drop=True)
)

sprint_wins_metrics

win_metrics = race_wins_metrics.merge(
    sprint_wins_metrics,
    on=['abbreviation', 'driver_number', 'full_name'],
    how='left'
)
win_metrics = win_metrics.sort_values(by='win_rate',ascending=False).reset_index(drop=True)
max_value_win_rate = win_metrics['win_rate'].max()
win_metrics['score_win_rate'] = higher_score(win_metrics['win_rate'],max_value_win_rate)
win_metrics['score_win_rate'] = win_metrics['score_win_rate'].round(2)
win_metrics

,abbreviation,driver_number,full_name,wins_count,races_participated,win_rate,sprint_wins_count,sprints_participated,sprint_win_rate,score_win_rate
0,VER,1,Max Verstappen,8,24,33.33,2,6,33.33,10.00
1,NOR,4,Lando Norris,7,24,29.17,2,6,33.33,8.75
2,PIA,81,Oscar Piastri,7,24,29.17,1,6,16.67,8.75
3,RUS,63,George Russell,2,24,8.33,0,6,0.00,2.50
4,ALB,23,Alexander Albon,0,24,0.00,0,6,0.00,0.00
5,BOR,5,Gabriel Bortoleto,0,24,0.00,0,6,0.00,0.00
6,BEA,87,Oliver Bearman,0,24,0.00,0,6,0.00,0.00
7,ANT,12,Kimi Antonelli,0,24,0.00,0,6,0.00,0.00
8,ALO,14,Fernando Alonso,0,24,0.00,0,6,0.00,0.00
9,COL,43,Franco Colapinto,0,17,0.00,0,4,0.00,0.00


In [ ]:
master_summary = drivers_performance.copy()
all_metrics = [finish_metrics,quali_metrics,quali_finish_gap_metrics,dnf_metrics,podium_metrics,win_metrics]
for metric in all_metrics:
    master_summary = master_summary.merge(
        metric,
        on=['abbreviation', 'driver_number', 'full_name'],
        how='left'
    )
master_summary.columns.tolist()

['abbreviation',
 'driver_number',
 'full_name',
 'total_points',
 'score_points',
 'races_avg_finish',
 'races_finish_std',
 'races_best_finish',
 'races_worst_finish',
 'races_finished',
 'sprint_avg_finish',
 'sprint_finish_std',
 'sprint_best_finish',
 'sprint_worst_finish',
 'sprint_races_finished',
 'score_races_avg_finish',
 'score_races_finish_std',
 'avg_quali_position',
 'best_quali_position',
 'worst_quali_position',
 'quali_sessions',
 'avg_sprint_quali',
 'best_sprint_quali',
 'worst_sprint_quali',
 'sprint_quali_sessions',
 'score_avg_quali_position',
 'avg_quali_race_gap',
 'races_used',
 'avg_sprint_quali_race_gap',
 'sprints_used',
 'score_avg_quali_race_gap',
 'race_entries',
 'dns_count',
 'dsq_count',
 'dnf_count',
 'races_started',
 'dnf_rate',
 'sprint_entries',
 'sprint_dns_count',
 'sprint_dsq_count',
 'sprint_dnf_count',
 'sprints_started',
 'sprint_dnf_rate',
 'score_dnf_rate',
 'podium_count',
 'races_participated_x',
 'podium_rate',
 'sprint_podium_count',
 

In [ ]:
master_summary['overall_score'] = (
    master_summary['score_points'] * 0.25 +
    master_summary['score_win_rate'] * 0.15 +
    master_summary['score_podium_rate'] * 0.15 +
    master_summary['score_races_avg_finish'] * 0.10 +
    master_summary['score_avg_quali_position'] * 0.10 +
    master_summary['score_dnf_rate'] * 0.15 +
    master_summary['score_races_finish_std'] * 0.05 +
    master_summary['score_avg_quali_race_gap'] * 0.05
).round(2)

master_summary['overall_rank'] = master_summary['overall_score'].rank(ascending=False, method='min')

master_summary[['full_name','abbreviation','total_points','overall_score','overall_rank']]


,abbreviation,driver_number,full_name,total_points,score_points,races_avg_finish,races_finish_std,races_best_finish,races_worst_finish,races_finished,...,score_podium_rate,wins_count,races_participated_y,win_rate,sprint_wins_count,sprints_participated_y,sprint_win_rate,score_win_rate,overall_score,overall_rank
0,NOR,4,Lando Norris,423.0,10.00,2.24,1.45,1.0,7.0,21,...,10.00,7,24,29.17,2,6,33.33,8.75,8.58,1.0
1,VER,1,Max Verstappen,421.0,9.95,3.04,2.51,1.0,10.0,23,...,8.33,8,24,33.33,2,6,33.33,10.00,8.48,2.0
2,PIA,81,Oscar Piastri,410.0,9.69,2.77,1.97,1.0,9.0,22,...,8.89,7,24,29.17,1,6,16.67,8.75,8.36,3.0
3,RUS,63,George Russell,319.0,7.54,4.54,2.50,1.0,11.0,24,...,5.00,2,24,8.33,0,6,0.00,2.50,6.23,4.0
4,LEC,16,Charles Leclerc,242.0,5.72,5.05,2.87,2.0,14.0,21,...,3.89,0,24,0.00,0,6,0.00,0.00,4.80,5.0
5,HAM,44,Lewis Hamilton,156.0,3.69,7.00,2.37,4.0,12.0,21,...,0.00,0,24,0.00,1,6,16.67,0.00,3.62,6.0
6,ANT,12,Kimi Antonelli,150.0,3.55,8.20,4.98,2.0,18.0,20,...,1.67,0,24,0.00,0,6,0.00,0.00,2.94,7.0
7,ALB,23,Alexander Albon,73.0,1.73,9.65,3.63,5.0,16.0,20,...,0.00,0,24,0.00,0,6,0.00,0.00,2.26,12.0
8,SAI,55,Carlos Sainz,64.0,1.51,10.42,3.89,3.0,18.0,19,...,1.16,0,23,0.00,0,6,0.00,0.00,2.11,14.0
9,ALO,14,Fernando Alonso,56.0,1.32,10.26,3.54,5.0,17.0,19,...,0.00,0,24,0.00,0,6,0.00,0.00,1.78,17.0


In [ ]:
final_columns = [

    'overall_rank',

    # Driver Identification
    'abbreviation',
    'full_name',
    'driver_number',

    # Overall Performance
    'total_points',
    
    # Main Race Performance
    'races_avg_finish',
    'races_finish_std',
    'races_best_finish',
    'races_worst_finish',
    
    # Sprint Performance
    'sprint_avg_finish',
    'sprint_finish_std',
    
    # Qualifying
    'avg_quali_position',
    
    # Qualifying vs Race Gap
    'avg_quali_race_gap',
    'avg_sprint_quali_race_gap',
    
    # Reliability
    'dnf_count',
    'dnf_rate',
    
    # Success Metrics
    'podium_count',
    'podium_rate',
    'wins_count',
    'win_rate',
    'sprint_podium_rate',
    'sprint_win_rate',

    #score
    'score_points',
    'score_win_rate',
    'score_podium_rate',
    'score_dnf_rate',
    'score_races_avg_finish',
    'score_avg_quali_position',
    'score_races_finish_std',
    'score_avg_quali_race_gap',
    
    'overall_score'
]
master_summary = master_summary[final_columns]
master_summary = master_summary.sort_values(by='overall_rank',ascending=True).reset_index(drop=True)
master_summary.head(22)

,overall_rank,abbreviation,full_name,driver_number,total_points,races_avg_finish,races_finish_std,races_best_finish,races_worst_finish,sprint_avg_finish,...,sprint_win_rate,score_points,score_win_rate,score_podium_rate,score_dnf_rate,score_races_avg_finish,score_avg_quali_position,score_races_finish_std,score_avg_quali_race_gap,overall_score
0,1.0,NOR,Lando Norris,4,423.0,2.24,1.45,1.0,7.0,3.20,...,33.33,10.00,8.75,10.00,7.50,8.58,8.26,7.09,2.11,8.58
1,2.0,VER,Max Verstappen,1,421.0,3.04,2.51,1.0,10.0,5.00,...,33.33,9.95,10.00,8.33,8.75,8.08,7.94,4.96,1.62,8.48
2,3.0,PIA,Oscar Piastri,81,410.0,2.77,1.97,1.0,9.0,1.75,...,16.67,9.69,8.75,8.89,8.75,8.25,8.21,6.04,0.69,8.36
3,4.0,RUS,George Russell,63,319.0,4.54,2.50,1.0,11.0,4.50,...,0.00,7.54,2.50,5.00,10.00,7.13,7.42,4.98,0.40,6.23
4,5.0,LEC,Charles Leclerc,16,242.0,5.05,2.87,2.0,14.0,6.40,...,0.00,5.72,0.00,3.89,7.50,6.81,6.79,4.24,1.69,4.80
5,6.0,HAM,Lewis Hamilton,44,156.0,7.00,2.37,4.0,12.0,7.83,...,16.67,3.69,0.00,0.00,7.50,5.58,4.68,5.24,5.60,3.62
6,7.0,ANT,Kimi Antonelli,12,150.0,8.20,4.98,2.0,18.0,7.83,...,0.00,3.55,0.00,1.67,5.00,4.82,4.95,0.00,1.51,2.94
7,8.0,HUL,Nico Hulkenberg,27,51.0,11.40,4.42,3.0,20.0,15.00,...,0.00,1.21,0.00,0.58,7.39,2.79,1.25,1.12,10.00,2.46
8,9.0,TSU,Yuki Tsunoda,22,33.0,12.30,3.11,6.0,17.0,8.00,...,0.00,0.78,0.00,0.00,8.75,2.23,2.60,3.76,3.03,2.33
9,10.0,OCO,Esteban Ocon,31,38.0,12.17,3.69,5.0,18.0,11.80,...,0.00,0.90,0.00,0.00,8.75,2.31,1.11,2.59,6.21,2.32


QUARTER-BY-QUARTER

In [ ]:
def quarter_by_quarter_calc(quali,races,sprint_quali,sprint):
    #total points by quarter
    if not sprint.empty:
        drivers_performance = pd.concat([races[['abbreviation','driver_number','full_name','points']],sprint[['abbreviation','driver_number','full_name','points']]],ignore_index = True)
    else:
        drivers_performance = races.copy()
        
    drivers_performance = (
        drivers_performance
        .groupby(['abbreviation', 'driver_number','full_name'], as_index=False)['points']
        .sum()
        .sort_values(by='points', ascending=False)
    )
    drivers_performance.rename(columns={'points':'total_points'},inplace=True)

    #finishing position metrics
    finish_data = races.copy()

    finish_data['race_position'] = pd.to_numeric(finish_data['race_position'],errors='coerce')


    finished_races = finish_data[~finish_data['status'].isin(['Retired','Did not start', 'Disqualified'])].copy()

    races_finish_metrics = (
        finished_races.groupby(['abbreviation', 'driver_number', 'full_name'],as_index=False)  
        .agg(
            races_avg_finish=('race_position', 'mean'),
            races_finish_std=('race_position', 'std'),
            races_best_finish=('race_position', 'min'),
            races_worst_finish=('race_position', 'max'),
            races_finished=('race_position', 'count')
        )
    )

    races_finish_metrics['races_avg_finish'] = races_finish_metrics['races_avg_finish'].round(2)
    races_finish_metrics['races_finish_std'] = races_finish_metrics['races_finish_std'].round(2)

    races_finish_metrics = races_finish_metrics.sort_values(
        by='races_avg_finish',
        ascending=True
    )
    
    if not sprint.empty:
        sprint_finish_data = sprint.copy()

        sprint_finish_data['sprint_position'] = pd.to_numeric(sprint_finish_data['sprint_position'],errors='coerce')

        sprint_finished_races = sprint_finish_data[~sprint_finish_data['status'].isin(['Retired','Did not start', 'Disqualified'])].copy()

        sprint_finish_metrics = sprint_finished_races.groupby(['abbreviation', 'driver_number', 'full_name'],as_index=False).agg(
            sprint_avg_finish=('sprint_position', 'mean'),
            sprint_finish_std=('sprint_position', 'std'),
            sprint_best_finish=('sprint_position', 'min'),
            sprint_worst_finish=('sprint_position', 'max'),
            sprint_races_finished=('sprint_position', 'count')
        )

        sprint_finish_metrics['sprint_avg_finish'] = sprint_finish_metrics['sprint_avg_finish'].round(2)
        sprint_finish_metrics['sprint_finish_std'] = sprint_finish_metrics['sprint_finish_std'].round(2)

        sprint_finish_metrics = sprint_finish_metrics.sort_values(
            by='sprint_avg_finish',
            ascending=True
        )

        finish_metrics = races_finish_metrics.merge(
            sprint_finish_metrics,
            on= ['abbreviation', 'driver_number', 'full_name'],
            how= 'left'
        )
        finish_metrics = finish_metrics.sort_values(by='races_avg_finish',ascending=True).reset_index(drop=True)
    else:
        finish_metrics = races_finish_metrics

    #QUALIFYING METRICS
    races_quali_data = quali.copy()
    races_quali_data['quali_position'] = pd.to_numeric(races_quali_data['quali_position'],errors='coerce')

    races_quali_metrics = (
        races_quali_data.groupby(['abbreviation', 'driver_number', 'full_name'],as_index=False)
        .agg(
            avg_quali_position = ('quali_position','mean'),
            best_quali_position=('quali_position', 'min'),
            worst_quali_position=('quali_position', 'max'),
            quali_sessions=('quali_position', 'count')
        )
    ) 

    races_quali_metrics['avg_quali_position'] = (races_quali_metrics['avg_quali_position'].round(2))

    races_quali_metrics = races_quali_metrics.sort_values(by='avg_quali_position',ascending=True).reset_index(drop=True)

    
    if not sprint_quali.empty:
        sprint_quali_data = sprint_quali.copy()
        sprint_quali_data['sprint_quali_position'] = pd.to_numeric(sprint_quali_data['sprint_quali_position'],errors='coerce') 

        sprint_quali_metrics = (
            sprint_quali_data.groupby(['abbreviation', 'driver_number', 'full_name'],as_index=False)
            .agg(
                avg_sprint_quali=('sprint_quali_position', 'mean'),
                best_sprint_quali=('sprint_quali_position', 'min'),
                worst_sprint_quali=('sprint_quali_position', 'max'),
                sprint_quali_sessions=('sprint_quali_position', 'count')
            )
        )
        sprint_quali_metrics= sprint_quali_metrics.sort_values(by='avg_sprint_quali',ascending=True).reset_index(drop=True)
        

        quali_metrics = races_quali_metrics.merge(
            sprint_quali_metrics,
            on=['abbreviation', 'driver_number', 'full_name'],
            how='left'
        )
        quali_metrics = quali_metrics.sort_values(by='avg_quali_position',ascending=True).reset_index(drop=True)
    else:
        quali_metrics = races_quali_metrics
    

    #QUALIFYING VS RACE PERFORMANCE GAP
    race_position_data = races.copy()

    # Convert positions to numeric
    race_position_data[['grid_position','race_position']] = race_position_data[['grid_position','race_position']].apply(pd.to_numeric,errors='coerce')

    # Use only drivers who officially finished
    race_position_data = race_position_data[race_position_data['status'] == 'Finished'].copy()

    # Use only drivers who officially finished
    race_position_data['quali_race_gap'] = race_position_data['grid_position'] - race_position_data['race_position']

    # Calculate metrics per driver
    races_finish_gap_metrics = race_position_data.groupby(
        ['abbreviation', 'driver_number', 'full_name'],as_index=False
        ).agg(
            avg_quali_race_gap = ('quali_race_gap','mean'),
            races_used = ('quali_race_gap','count')
            )
    races_finish_gap_metrics['avg_quali_race_gap'] = races_finish_gap_metrics['avg_quali_race_gap'].round(2)

    races_finish_gap_metrics = races_finish_gap_metrics.sort_values(
            by='avg_quali_race_gap',
            ascending=False
        ).reset_index(drop=True)

    if not sprint.empty:
        sprint_position_data = sprint.copy()

        # Convert positions to numeric
        sprint_position_data[['grid_position', 'sprint_position']] = (sprint_position_data[['grid_position', 'sprint_position']].apply(pd.to_numeric, errors='coerce'))

        # Use only drivers who officially finished
        sprint_position_data = sprint_position_data[sprint_position_data['status'] == 'Finished'].copy()

        # Calculate positions gained/lost
        sprint_position_data['sprint_quali_race_gap'] = (sprint_position_data['grid_position'] - sprint_position_data['sprint_position'])

        # Calculate metrics per driver
        sprint_finish_gap_metrics = (
            sprint_position_data.groupby(['abbreviation', 'driver_number', 'full_name'],as_index=False
            ).agg(
                avg_sprint_quali_race_gap=('sprint_quali_race_gap', 'mean'),
                sprints_used=('sprint_quali_race_gap', 'count')
            )
        )
        sprint_finish_gap_metrics['avg_sprint_quali_race_gap'] = sprint_finish_gap_metrics['avg_sprint_quali_race_gap'].round(2)
        # Sort by biggest average positions gained
        sprint_finish_gap_metrics = (sprint_finish_gap_metrics.sort_values(
                by='avg_sprint_quali_race_gap',
                ascending=False
            )
            .reset_index(drop=True)
        )


        quali_finish_gap_metrics = races_finish_gap_metrics.merge(
            sprint_finish_gap_metrics,
            on=['abbreviation', 'driver_number', 'full_name'],
            how='left'
        )
        quali_finish_gap_metrics = quali_finish_gap_metrics.sort_values(by = 'avg_quali_race_gap',ascending=False).reset_index(drop=True)
    else:
        quali_finish_gap_metrics = races_finish_gap_metrics

    
    #DNF (DID NOT FINISH) METRICS
    races_dnf_data = races.copy()

    #mark DNFs
    races_dnf_data['dnf'] = (races_dnf_data['status'] == 'Retired').astype(int)
    #count dns and dsq
    races_dnf_metrics = races_dnf_data.groupby(
        ['abbreviation', 'driver_number', 'full_name'],as_index=False
        ).agg(
            race_entries = ('status','size'),
            dns_count = ('status', lambda x: (x == 'Did not start').sum()),
            dsq_count = ('status', lambda x: (x == 'Disqualified').sum()),
            dnf_count = ('dnf','sum')

        )

    races_dnf_metrics['races_started'] = (
        races_dnf_metrics['race_entries'] - races_dnf_metrics['dns_count']
    )

    races_dnf_metrics['dnf_rate'] = (
        races_dnf_metrics['dnf_count'] / races_dnf_metrics['races_started']
    ) * 100
    races_dnf_metrics['dnf_rate'] = races_dnf_metrics['dnf_rate'].round(2)
    races_dnf_metrics = races_dnf_metrics.sort_values(by='dnf_rate',ascending=True).reset_index(drop=True)


    # DNF (Did Not Finish) metrics - SPRINT RACES
    if not sprint.empty:
        sprint_dnf_data = sprint.copy()

        # Mark DNFs
        sprint_dnf_data['dnf'] = (sprint_dnf_data['status'] == 'Retired').astype(int)


        # Calculate Sprint DNF metrics
        sprint_dnf_metrics = (
            sprint_dnf_data.groupby(['abbreviation', 'driver_number', 'full_name'],as_index=False
            )
            .agg(
                sprint_entries=('status', 'size'),
                sprint_dns_count=('status',lambda x: (x == 'Did not start').sum()),
                sprint_dsq_count=('status',lambda x: (x == 'Disqualified').sum()),
                sprint_dnf_count=('dnf', 'sum')
            )
        )

        # Calculate Sprint races started
        sprint_dnf_metrics['sprints_started'] = (
            sprint_dnf_metrics['sprint_entries'] - sprint_dnf_metrics['sprint_dns_count']
        )


        # Calculate DNF rate
        sprint_dnf_metrics['sprint_dnf_rate'] = (
            sprint_dnf_metrics['sprint_dnf_count'] / sprint_dnf_metrics['sprints_started']
        ) * 100


        # Round DNF rate
        sprint_dnf_metrics['sprint_dnf_rate'] = (sprint_dnf_metrics['sprint_dnf_rate'].round(2))


        # Sort by lowest DNF rate
        sprint_dnf_metrics = (sprint_dnf_metrics.sort_values(by='sprint_dnf_rate', ascending=True)
            .reset_index(drop=True)
        )


        dnf_metrics = races_dnf_metrics.merge(
            sprint_dnf_metrics,
            on=['abbreviation', 'driver_number', 'full_name'],
            how='left'
        )
        dnf_metrics = dnf_metrics.sort_values(by='dnf_rate',ascending=True).reset_index(drop=True)
    else:
        dnf_metrics = races_dnf_metrics


    #PODIUM METRICS
    race_podium_data = races.copy()

    race_podium_data['race_position'] = pd.to_numeric(race_podium_data['race_position'],errors='coerce')
    race_podium_data['podium_count'] = (race_podium_data['race_position']<=3).astype(int)


    race_podium_metrics = race_podium_data.groupby(
        ['abbreviation', 'driver_number', 'full_name'],as_index=False
        ).agg(
            podium_count = ('podium_count','sum'),
            race_entries = ('status','size'),
            dns_count = ('status', lambda x: (x == 'Did not start').sum()),
        )
    #removes DNS from the races
    race_podium_metrics['races_participated'] = race_podium_metrics['race_entries'] - race_podium_metrics['dns_count']

    #podium rate
    race_podium_metrics['podium_rate'] = (race_podium_metrics['podium_count']/race_podium_metrics['races_participated'])*100

    race_podium_metrics.drop(
        columns=['race_entries', 'dns_count'],
        inplace=True
    )

    race_podium_metrics['podium_rate'] = race_podium_metrics['podium_rate'].round(2)
    race_podium_metrics = race_podium_metrics.sort_values(by='podium_rate',ascending=False).reset_index(drop=True)


    #Sprint Podium Metrics
    if not sprint.empty:
        sprint_podium_data = sprint.copy()

        sprint_podium_data['sprint_position'] = pd.to_numeric(sprint_podium_data['sprint_position'],errors='coerce')

        # Mark podiums
        sprint_podium_data['sprint_podium_count'] = (sprint_podium_data['sprint_position'] <= 3).astype(int)

        # Calculate metrics
        sprint_podium_metrics = sprint_podium_data.groupby(
            ['abbreviation', 'driver_number', 'full_name'],as_index=False
        ).agg(
            sprint_podium_count=('sprint_podium_count', 'sum'),
            sprint_entries=('status', 'size'),
            sprint_dns_count=('status',lambda x: (x == 'Did not start').sum())
        )

        # Remove DNS
        sprint_podium_metrics['sprints_participated'] = (sprint_podium_metrics['sprint_entries'] - sprint_podium_metrics['sprint_dns_count'])

        # Podium rate
        sprint_podium_metrics['sprint_podium_rate'] = (sprint_podium_metrics['sprint_podium_count'] / sprint_podium_metrics['sprints_participated']) * 100

        # Round
        sprint_podium_metrics['sprint_podium_rate'] = (sprint_podium_metrics['sprint_podium_rate'].round(2))

        # Remove unnecessary columns
        sprint_podium_metrics.drop(
            columns=['sprint_entries', 'sprint_dns_count'],
            inplace=True
        )

        # Sort
        sprint_podium_metrics = (
            sprint_podium_metrics.sort_values(by='sprint_podium_rate', ascending=False).reset_index(drop=True)
        )


        podium_metrics = race_podium_metrics.merge(
            sprint_podium_metrics,
            on=['abbreviation', 'driver_number', 'full_name'],
            how='left'
        )
        podium_metrics = podium_metrics.sort_values(by='podium_rate',ascending=False).reset_index(drop=True)
    else:
        podium_metrics = race_podium_metrics

    
    #WIN METRICCS
    race_wins_data = races.copy()

    race_wins_data['race_position'] = pd.to_numeric(race_wins_data['race_position'],errors='coerce')

    race_wins_data['win_count'] = (race_wins_data['race_position'] == 1).astype(int)

    race_wins_metrics = race_wins_data.groupby(
        ['abbreviation', 'driver_number', 'full_name'],as_index=False
        ).agg(
            wins_count = ('win_count','sum'),
            race_entries = ('status','size'),
            dns_count = ('status', lambda x: (x == 'Did not start').sum())
        )
    race_wins_metrics['races_participated'] = race_wins_metrics['race_entries'] - race_wins_metrics['dns_count']

    race_wins_metrics['win_rate'] = (race_wins_metrics['wins_count'] / race_wins_metrics['races_participated']) * 100

    race_wins_metrics.drop(columns=['race_entries', 'dns_count'],inplace=True)

    race_wins_metrics['win_rate'] = race_wins_metrics['win_rate'].round(2)

    race_wins_metrics = race_wins_metrics.sort_values(by='win_rate',ascending=False).reset_index(drop=True)


    # Sprint Win Metrics
    if not sprint.empty:
        sprint_wins_data = sprint.copy()

        # Convert position to numeric
        sprint_wins_data['sprint_position'] = pd.to_numeric(sprint_wins_data['sprint_position'],errors='coerce')

        # Mark wins
        sprint_wins_data['sprint_win_count'] = (sprint_wins_data['sprint_position'] == 1).astype(int)


        # Calculate metrics
        sprint_wins_metrics = sprint_wins_data.groupby(
            ['abbreviation', 'driver_number', 'full_name'],as_index=False
        ).agg(
            sprint_wins_count=('sprint_win_count', 'sum'),
            sprint_entries=('status', 'size'),
            sprint_dns_count=('status',lambda x: (x == 'Did not start').sum())
        )


        # Calculate sprints participated
        sprint_wins_metrics['sprints_participated'] = (sprint_wins_metrics['sprint_entries'] - sprint_wins_metrics['sprint_dns_count'])


        # Calculate win rate
        sprint_wins_metrics['sprint_win_rate'] = (sprint_wins_metrics['sprint_wins_count'] / sprint_wins_metrics['sprints_participated']) * 100


        # Remove unnecessary columns
        sprint_wins_metrics.drop(
            columns=['sprint_entries', 'sprint_dns_count'],inplace=True
        )


        # Round win rate
        sprint_wins_metrics['sprint_win_rate'] = (
            sprint_wins_metrics['sprint_win_rate'].round(2)
        )


        # Sort by highest win rate
        sprint_wins_metrics = (
            sprint_wins_metrics.sort_values(by='sprint_win_rate', ascending=False).reset_index(drop=True)
        )

        win_metrics = race_wins_metrics.merge(
            sprint_wins_metrics,
            on=['abbreviation', 'driver_number', 'full_name'],
            how='left'
        )
        win_metrics = win_metrics.sort_values(by='win_rate',ascending=False).reset_index(drop=True)
    else:
        win_metrics = race_wins_metrics

    #COMBINE ALL METRICS
    quarter_summary = drivers_performance.copy()
    all_metrics = [finish_metrics,quali_metrics,quali_finish_gap_metrics,dnf_metrics,podium_metrics,win_metrics]
    for metric in all_metrics:
        quarter_summary = quarter_summary.merge(
            metric,
            on=['abbreviation', 'driver_number', 'full_name'],
            how='left'
        )
    final_columns = [
        # Driver Identification
        'abbreviation',
        'full_name',
        'driver_number',
        
        # Overall Performance
        'total_points',
        
        # Main Race Performance
        'races_avg_finish',
        'races_finish_std',
        'races_best_finish',
        'races_worst_finish',
        
        # Sprint Performance
        'sprint_avg_finish',
        'sprint_finish_std',
        
        # Qualifying
        'avg_quali_position',
        
        # Qualifying vs Race Gap
        'avg_quali_race_gap',
        'avg_sprint_quali_race_gap',
        
        # Reliability
        'dnf_count',
        'dnf_rate',
        
        # Success Metrics
        'podium_count',
        'podium_rate',
        'wins_count',
        'win_rate',
        'sprint_podium_rate',
        'sprint_win_rate'
    ]
    for column in final_columns:
        if column not in quarter_summary.columns:
            quarter_summary[column] = pd.NA
    quarter_summary = quarter_summary[final_columns]
    return quarter_summary

In [ ]:
quarters = {
    'Q1' : range(1,7),
    'Q2': range(7, 13),
    'Q3': range(13, 19),
    'Q4': range(19, 25)
}

quarter_summaries ={}
for quarter, rounds in quarters.items():

    quarter_quali = quali_df[quali_df['race_number'].isin(rounds)]

    quarter_races = races_df[races_df['race_number'].isin(rounds)]

    quarter_sprint_quali = sprint_quali_df[sprint_quali_df['race_number'].isin(rounds)]

    quarter_sprint = sprint_df[sprint_df['race_number'].isin(rounds)]
    # print(quarter_sprint.shape)
    # print(quarter_sprint[['race_number', 'race_name']])
    #calculate metrics fo every quarter
    quarter_summaries[quarter] = quarter_by_quarter_calc(quarter_quali,quarter_races,quarter_sprint_quali,quarter_sprint)

for quarter,df in quarter_summaries.items():
    df['quarter'] = quarter

In [ ]:
master_summary.to_csv('data/2025_master_summary.csv',index=False)

for quarter, df in quarter_summaries.items():
    df.to_csv(f'data/2025_{quarter}_summary.csv',index=False)

quarterly_summary = pd.concat(quarter_summaries.values(), ignore_index=True)

quarterly_summary.to_csv('data/2025_quarterly_summary.csv', index=False)

C:\Users\manid\AppData\Local\Temp\ipykernel_2000\3275750108.py:6: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  quarterly_summary = pd.concat(quarter_summaries.values(), ignore_index=True)
